# Receiver benchmark: M0–M3

Run the receiver benchmark first using [README.md](README.md), then run all cells.
This notebook reads the saved CSVs and manifest. It does not load checkpoints,
train models, or measure new timings.

- **M0:** Teacher receiver
- **M1:** CE-only Student
- **M2:** Combined KD Student
- **M3:** Liu receiver-side adaptation

Latency refers to the benchmark's fixed number of greedy decoding steps on
shared synthetic inputs. It does not measure reconstruction quality or
end-to-end transmission latency.


In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Set to a run directory containing summary.csv, latency_samples.csv and
# benchmark_manifest.json. None picks the newest complete report by file mtime.
REPORT_DIR = None
# Example: REPORT_DIR = "results/article_kd/receiver_benchmark/seed42-b1-src32-steps24"

# Optional new directory for PNG/PDF exports; None displays plots only.
EXPORT_DIR = None

MODELS = ["M0", "M1", "M2", "M3"]
COLORS = dict(zip(MODELS, ["#4477AA", "#228833", "#AA3377", "#CC6600"]))
FILES = ("summary.csv", "latency_samples.csv", "benchmark_manifest.json")

def find_repo(start):
    for path in (start, *start.parents):
        if (path / "benchmarks/receiver/runner.py").is_file():
            return path
    raise FileNotFoundError("Open Jupyter from the pecSC repository or a folder inside it.")

REPO = find_repo(Path.cwd().resolve())
if REPORT_DIR is None:
    root = REPO / "results/article_kd/receiver_benchmark"
    choices = [p.parent for p in root.glob("*/benchmark_manifest.json")
               if all((p.parent / name).is_file() for name in FILES)]
    if not choices:
        raise FileNotFoundError(f"No complete benchmark report under {root}. Run the benchmark or set REPORT_DIR.")
    report_dir = max(choices, key=lambda p: ((p / FILES[2]).stat().st_mtime_ns, str(p)))
else:
    report_dir = (REPO / Path(REPORT_DIR).expanduser()).resolve()

print(f"Report: {report_dir}")


The loader checks model coverage, repeat counts and the saved latency
statistics before plotting. All four rows must come from the same report.


In [ ]:
summary = pd.read_csv(report_dir / FILES[0])
samples = pd.read_csv(report_dir / FILES[1])
manifest = json.loads((report_dir / FILES[2]).read_text(encoding="utf-8"))
workload = manifest["workload"]
runtime = manifest["runtime"]

def check_columns(frame, expected, filename):
    missing = set(expected) - set(frame.columns)
    if missing:
        raise ValueError(f"{filename}: missing columns {sorted(missing)}")

check_columns(summary, ["model", "receiver_parameters", "receiver_checkpoint_bytes",
                        "receiver_parameter_reduction_pct_vs_m0", "batch_size",
                        "decode_steps", "samples", "median_batch_latency_ms",
                        "p95_batch_latency_ms"], FILES[0])
check_columns(samples, ["model", "repeat", "batch_latency_ms"], FILES[1])
if len(summary) != len(MODELS) or set(summary["model"]) != set(MODELS):
    raise ValueError("summary.csv must contain exactly one row per model M0–M3.")
if set(samples["model"]) != set(MODELS):
    raise ValueError("latency_samples.csv must contain samples for all four models.")
if not np.isfinite(samples["batch_latency_ms"]).all() or (samples["batch_latency_ms"] <= 0).any():
    raise ValueError("Timing samples must be finite and positive.")

summary = summary.set_index("model").loc[MODELS]
for column in ("batch_size", "decode_steps"):
    if not summary[column].eq(workload[column]).all():
        raise ValueError(f"Summary and manifest disagree on {column}.")

repeats = workload["repeats_per_model"]
for model in MODELS:
    rows = samples.loc[samples["model"].eq(model)]
    if summary.loc[model, "samples"] != repeats or sorted(rows["repeat"]) != list(range(1, repeats + 1)):
        raise ValueError(f"{model}: missing or duplicate timing repeats.")
    values = np.sort(rows["batch_latency_ms"].to_numpy())
    expected = [np.median(values), values[math.ceil(0.95 * repeats) - 1]]
    recorded = summary.loc[model, ["median_batch_latency_ms", "p95_batch_latency_ms"]].to_numpy(dtype=float)
    if not np.allclose(expected, recorded, rtol=1e-8, atol=1e-10):
        raise ValueError(f"{model}: summary timings disagree with raw samples.")

device = runtime.get("gpu") or runtime["device"]
caption = (f"{device} · batch {workload['batch_size']} · source length "
           f"{workload['source_length']} · {workload['decode_steps']} decoding steps · {repeats} repeats")
print(caption)

table = summary[["receiver_parameters", "receiver_parameter_reduction_pct_vs_m0",
                 "median_batch_latency_ms", "p95_batch_latency_ms"]].copy()
table.insert(1, "checkpoint_MiB", summary["receiver_checkpoint_bytes"] / 1024**2)
display(table.round(3))


In [ ]:
colors = [COLORS[name] for name in MODELS]
fig_size, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].bar(MODELS, summary["receiver_parameters"] / 1e6, color=colors)
axes[0].set(title="Receiver parameters", ylabel="Parameters (millions)")
axes[1].bar(MODELS, table["checkpoint_MiB"], color=colors)
axes[1].set(title="Serialized receiver checkpoint", ylabel="Size (MiB)")
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
fig_size.suptitle("Receiver size by model")
plt.show()


In [ ]:
fig_latency, axes = plt.subplots(1, 2, figsize=(12, 4.6), layout="constrained")
x = np.arange(len(MODELS))
width = 0.34
axes[0].bar(x - width / 2, summary["median_batch_latency_ms"], width, label="Median", color="#4477AA")
axes[0].bar(x + width / 2, summary["p95_batch_latency_ms"], width, label="p95", color="#CC6600")
axes[0].set(xticks=x, xticklabels=MODELS, title="Latency summary", ylabel="Milliseconds per batch")
axes[0].legend()
for model in MODELS:
    run = samples.loc[samples["model"].eq(model)].sort_values("repeat")
    axes[1].plot(run["repeat"], run["batch_latency_ms"], marker="o", markersize=3,
                 linewidth=1, color=COLORS[model], label=model)
axes[1].set(title="Timing by repeat", xlabel="Repeat", ylabel="Milliseconds per batch")
axes[1].legend(ncol=2)
for ax in axes:
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
fig_latency.suptitle(caption, fontsize=10)
plt.show()


At batch size 1, batch latency is latency for one input. M1–M3 have the same
receiver architecture, so expect equal parameter counts; small latency differences
alone do not establish an effect of the training objective.


In [ ]:
if EXPORT_DIR is not None:
    export_dir = (REPO / Path(EXPORT_DIR).expanduser()).resolve()
    export_dir.mkdir(parents=True, exist_ok=False)
    for name, figure in (("receiver_size", fig_size), ("receiver_latency", fig_latency)):
        figure.savefig(export_dir / f"{name}.png", dpi=180)
        figure.savefig(export_dir / f"{name}.pdf")
    print(f"Figures saved to {export_dir}")
